In [1]:
import cv2
import mediapipe as mp
import numpy as np
import joblib

# Carregar os modelos guardados no Grupo12_2.ipynb
mlp = joblib.load("models/mlp_asl_landmarks.joblib")
scaler = joblib.load("models/scaler_asl_landmarks.joblib")
label_encoder = joblib.load("models/label_encoder_asl_landmarks.joblib")

# Configurar o MediaPipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False, # False porque é vídeo em tempo real
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

/home/tomas/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
2026-03-27 14:06:58.250257: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 14:06:58.285358: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-27 14:06:58.446272: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-27 14:06:58.447307: I tensorflow/core/platform/cpu_f

In [2]:
# def normalize_landmarks(lms):
#     # 1. Referência ao pulso (ponto 0)
#     wx, wy, wz = lms[0].x, lms[0].y, lms[0].z

#     # 2. Cálculo da escala (distância média do pulso às bases dos dedos)
#     base_ids = [5, 9, 13, 17]
#     dists = []
#     for idx in base_ids:
#         dx = lms[idx].x - wx
#         dy = lms[idx].y - wy
#         dz = lms[idx].z - wz
#         dists.append((dx**2 + dy**2 + dz**2) ** 0.5)

#     scale = float(np.mean(dists))
    
#     # Prevenção de erro
#     if scale < 1e-6:
#         return None

#     # 3. Normalização
#     features = []
#     for lm in lms:
#         features.extend([
#             (lm.x - wx) / scale,
#             (lm.y - wy) / scale,
#             (lm.z - wz) / scale
#         ])
#     return features



def normalize_landmarks(lms):
    # 1. Referência ao pulso (ponto 0)
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z

    # 2. Cálculo da escala robusta
    base_ids = [5, 9, 13, 17]
    dists_base = []
    for idx in base_ids:
        dx = lms[idx].x - wx
        dy = lms[idx].y - wy
        dz = lms[idx].z - wz
        dists_base.append((dx**2 + dy**2 + dz**2) ** 0.5)

    scale = float(np.mean(dists_base))
    if scale < 1e-6: 
        return None

    features = []
    
    # 3. Coordenadas (63 valores)
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])
    
    # 4. NOVOS ATRIBUTOS: Distâncias (21 valores) - ESSENCIAL PARA O NOVO MODELO
    for lm in lms:
        d = ((lm.x - wx)**2 + (lm.y - wy)**2 + (lm.z - wz)**2)**0.5
        features.append(d / scale)

    return features

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [ ]:
# Ligar a webcam
cap = cv2.VideoCapture(0)

current_letter = ""
consecutive_frames = 0
CONFIDENCE_THRESHOLD = 5 # Apenas muda a letra se a mesma previsão ocorrer 5 vezes seguidas

print("Pressione a tecla 'q' na janela de vídeo para sair.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Não foi possível aceder à webcam.")
        break

    # Espelhar a imagem para ser como um espelho e converter para RGB
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Processar com MediaPipe
    results = hands.process(rgb_frame)
    
    predicted_label = ""

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # 1. Desenhar os pontos na mão para feedback visual
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # 2. Extrair e normalizar
            features = normalize_landmarks(hand_landmarks.landmark)
            
            if features is not None:
                # 3. Preparar os dados para o modelo (reshape e scale)
                features_array = np.array(features).reshape(1, -1)
                features_scaled = scaler.transform(features_array)
                
                # 4. Fazer a previsão
                prediction = mlp.predict(features_scaled)
                predicted_label = label_encoder.inverse_transform(prediction)[0]
                
    # 5. Lógica de suavização (smoothing)
    if predicted_label == current_letter and predicted_label != "":
        consecutive_frames += 1
    else:
        consecutive_frames = 0
        current_letter = predicted_label
        
    # Só mostra se tivermos alguma confiança (várias frames seguidas)
    display_text = current_letter if consecutive_frames >= CONFIDENCE_THRESHOLD else "..."
    
    # Escrever no ecrã
    cv2.putText(frame, f"Gesto: {display_text}", (10, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3, cv2.LINE_AA)

    # Mostrar a imagem
    cv2.imshow("Tradutor ASL - Grupo 12", frame)

    # Pressionar 'q' para fechar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Limpar tudo ao fechar
cap.release()
cv2.destroyAllWindows()

W0000 00:00:1774620420.767763   23866 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774620420.785870   23877 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Pressione a tecla 'q' na janela de vídeo para sair.


W0000 00:00:1774620421.595071   23871 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
/home/tomas/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/tomas/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/tomas/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/tomas/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/

KeyboardInterrupt: 

: 